# VPython in a GitHub Codespace — desktop VS Code probe

**Question under test:** does VS Code's automatic port forwarding make the
websocket-only frontend work unchanged when the kernel runs in a codespace
and desktop VS Code is attached?

Setup expected in the codespace: `pip install ipykernel` + the
`feat/vscode-frontend` branch of vpython-jupyter; the VPython extension
(>= 0.1.4) installed in VS Code.

Run the cells in order. **Watch the Ports view** (Terminal panel → Ports):
after cell 2 a forwarded port should appear (vpython's tornado server).
The scene renders under cell 2 (the import cell owns the container).

In [ ]:
# Cell 1: environment diagnostics BEFORE importing vpython.
# Tells us whether ws-frontend auto-detection works for remote kernels
# or needs a CODESPACES clause.
import os
for var in ('CODESPACES', 'CODESPACE_NAME', 'VSCODE_PID', 'VSCODE_CWD',
            'VSCODE_IPC_HOOK_CLI', 'VPYTHON_FRONTEND'):
    print(f'{var} = {os.environ.get(var)!r}')

if os.environ.get('VSCODE_PID') or os.environ.get('VSCODE_CWD'):
    print('\nauto-detection would choose the ws frontend: no override needed')
else:
    os.environ['VPYTHON_FRONTEND'] = 'ws'
    print('\nauto-detection would MISS (no VSCODE_* vars in remote kernel):\n'
          'forcing VPYTHON_FRONTEND=ws -> detection needs a CODESPACES clause')

In [ ]:
# Cell 2: import announces the port; the renderer must reach it through
# the forwarded tunnel. If nothing renders within ~30 s, the import
# raises with a diagnostic (PR #292) — copy that error into the report.
from vpython import *
print('vpython imported — scene container above/below this line')

In [ ]:
# Cell 3: static scene — proves the downlink.
floor = box(pos=vec(0, -1, 0), size=vec(6, 0.2, 6), color=color.green)
ball = sphere(pos=vec(0, 2, 0), radius=0.4, color=color.red, make_trail=True)
print('objects created — floor + ball should be visible')

In [ ]:
# Cell 4: animation through the tunnel — proves sustained throughput.
ball.v = vec(0.4, 0, 0.2)
g, dt = vec(0, -9.8, 0), 0.01
for _ in range(1000):
    rate(100)
    ball.v = ball.v + g * dt
    ball.pos = ball.pos + ball.v * dt
    if ball.pos.y - ball.radius < floor.pos.y + 0.1:
        ball.v.y = -ball.v.y * 0.9
print('done — was the animation smooth? try zoom/orbit (uplink check)')

## Report

1. Cell 1: which env vars were set? was the override needed?
2. Cell 2: did a port appear in the Ports view? did the scene container render?
3. Cells 3–4: objects visible? animation smooth? zoom/orbit responsive?
4. If it failed: the #292 timeout error text + whether manually forwarding
   the announced port (Ports view → Forward a Port) and re-running fixes it.